# Import

In [19]:
pip install sastrawi wordcloud keybert rapidfuzz yake

In [20]:
# Basic
import pandas as pd  # Manipulasi dan analisis data
import numpy as np  # Komputasi numerik
import datetime as dt  # Manipulasi waktu dan tanggal
import re  # Ekspresi reguler untuk pembersihan teks
import string  # Konstanta string seperti tanda baca

# Konfigurasi opsional
pd.options.mode.chained_assignment = None  # Menonaktifkan warning chaining assignment
seed = 0
np.random.seed(seed)  # Menetapkan seed untuk reproducibility


# Text Preprocessing
import nltk  # Natural Language Toolkit (NLP)
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Stopwords untuk berbagai bahasa
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')  # Untuk mendukung versi baru nltk

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from keybert import KeyBERT  # Ekstraksi kata kunci berbasis BERT
from rapidfuzz import fuzz

# Visualization
import matplotlib.pyplot as plt  # Visualisasi dasar
import seaborn as sns  # Visualisasi statistik
from wordcloud import WordCloud  # Membuat Word Cloud


# Feature Engineering
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF untuk representasi teks
from sklearn.preprocessing import StandardScaler  # Normalisasi data numerik
import yake

# Training & Evaluation
from sklearn.model_selection import train_test_split  # Membagi dataset menjadi train dan test
from sklearn.metrics import accuracy_score, classification_report  # Evaluasi performa model

# Algorithm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Data Assess

In [21]:
df = pd.read_csv('raw_review.csv')
df

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,54182fbf-0f32-4359-a910-b781c8232870,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ok,5,0,1.2.6.1961,2025-08-18 23:24:07,NaN,NaN,1.2.6.1961
1,23eba42b-b9a7-420f-b564-2eed9990daad,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,developer tolol soal sistem matching. Sistemny...,1,277,1.2.6.1961,2025-08-18 23:16:38,NaN,NaN,1.2.6.1961
2,49329697-42df-45a9-9d30-36cb0d450560,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,gamenya asik dah buruan cobain sekarang gameny...,5,0,NaN,2025-08-18 22:58:10,NaN,NaN,NaN
3,a97e8026-6e9f-4e09-8dd0-f22a9c00620a,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,game pembodohan,1,0,1.2.6.1961,2025-08-18 22:34:36,NaN,NaN,1.2.6.1961
4,9362e924-4a50-40c8-9c51-bc5f5a29cd75,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,SERU MANTAP KING,5,0,1.2.6.1961,2025-08-18 22:21:36,NaN,NaN,1.2.6.1961
...,...,...,...,...,...,...,...,...,...,...,...
11621,886fad6d-2671-44aa-a641-f5f2e1d37938,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,update di play store di dalam masih update gim...,1,0,1.1.85.1742,2025-05-28 00:34:34,NaN,NaN,1.1.85.1742
11622,fc602530-d6a2-4663-ad3b-05c8b8c7cd62,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Strategi yang rumit tapi seru dan menyenangkan...,5,0,1.1.77.1651,2025-05-28 00:33:33,NaN,NaN,1.1.77.1651
11623,5b22dee8-1ac9-4b49-b03c-86e06731a8e1,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,mantab ....,5,0,1.1.77.1651,2025-05-28 00:24:10,NaN,NaN,1.1.77.1651
11624,c449ed61-72f0-4964-8fde-3fc649b7af74,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ok,5,0,1.1.68.1562,2025-05-28 00:12:04,NaN,NaN,1.1.68.1562


In [22]:
df = df[['content', 'score']]
df

,content,score
0,ok,5
1,developer tolol soal sistem matching. Sistemny...,1
2,gamenya asik dah buruan cobain sekarang gameny...,5
3,game pembodohan,1
4,SERU MANTAP KING,5
...,...,...
11621,update di play store di dalam masih update gim...,1
11622,Strategi yang rumit tapi seru dan menyenangkan...,5
11623,mantab ....,5
11624,ok,5


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11626 entries, 0 to 11625
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  11626 non-null  object
 1   score    11626 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 181.8+ KB


In [24]:
print(df.duplicated().sum())

2115


In [25]:
df = df.drop_duplicates(subset=['content']).reset_index(drop=True)
df

,content,score
0,ok,5
1,developer tolol soal sistem matching. Sistemny...,1
2,gamenya asik dah buruan cobain sekarang gameny...,5
3,game pembodohan,1
4,SERU MANTAP KING,5
...,...,...
9401,terlalu sering eror saat game.. seperti frame ...,1
9402,update di play store di dalam masih update gim...,1
9403,Strategi yang rumit tapi seru dan menyenangkan...,5
9404,mantab ....,5


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9406 entries, 0 to 9405
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  9406 non-null   object
 1   score    9406 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 147.1+ KB


# Preprocessing

In [27]:
kamus_alay_url = 'https://raw.githubusercontent.com/ezaaputra/Sentiment-Analysis-Using-BERT/main/kamus_alay.csv'
kamus_alay = pd.read_csv(kamus_alay_url)

# Ubah menjadi dictionary
normalize_word_dict = {row[0]: row[1] for _, row in kamus_alay.iterrows()}

slangwords = {
    'sebum': 'sebelum', 'rilus': 'rilis', 'bljr': 'belajar', 'dpet': 'dapat',
    'bnr': 'benar', 'jngn': 'jangan', 'kaya': 'seperti', 'hoki': 'beruntung',
    'nda': 'tidak', 'legen': 'legend', 'ko': 'kok', 'ntap': 'mantap',
    'mantuaf': 'mantap', 'gw': 'saya', 'samapi': 'sampai', 'bnyak': 'banyak',
    'gx': 'tidak', 'ngk': 'tidak', 'gak': 'tidak', 'ga': 'tidak', 'gk': 'tidak',
    'udah': 'sudah', 'udh': 'sudah', 'sdh': 'sudah', 'yg': 'yang', 'aj': 'saja',
    'aja': 'saja', 'nma': 'nama', 'jd': 'menjadi', 'skrng': 'sekarang',
    'skrg': 'sekarang', 'kalo': 'kalau', 'klo': 'kalau', 'kl': 'kalau',
    'klau': 'kalau', 'pake': 'pakai', 'trus': 'terus', 'trs': 'terus',
    'tetep': 'tetap', 'ttp': 'tetap', 'bnget': 'sekali', 'banget': 'sekali',
    'bgt': 'sekali', 'bkin': 'membuat', 'bikin': 'membuat', 'dpt': 'dapat',
    'dr': 'dari', 'dri': 'dari', 'ya': '', 'sih': '', 'dong': '', 'lemot': 'lambat',
    'no': 'nomor', 'hp': 'ponsel', 'apk': 'aplikasi', 'aku': 'saya',
    'bgtu': 'begitu', 'bbrp': 'beberapa', 'sm': 'sama', 'tp': 'tapi',
    'pdhl': 'padahal', 'krn': 'karena', 'kren': 'keren', 'dg': 'dengan',
    'dgn': 'dengan', 'jg': 'juga', 'mlh': 'malah', 'mnt': 'menit', 'ngbug': 'bug',
    'ngelag': 'lag', 'nglag': 'lag', 'seeru': 'seru', 'p2w': 'pay to win', 'pls': 'tolong',
    'hoki': 'untung', 'cmd': 'commander', 'b1': 'bintang satu', 'b2': 'bintang dua',
    'b3': 'bintang tiga', 's1': 'season satu', 's2': 'season dua', 's3': 'season tiga',
    's4': 'season empat', 'mantab': 'mantap', 'system': 'sistem', 'gem': 'game',
    'gabut': 'bosan', 'ngeleg': 'lag', 'muantap': 'mantap', 'mantul': 'mantap',
    'mantabz': 'mantap', 'jaringan': 'koneksi', 'gaca': 'gacha', 'sangattttt': 'sangat',
    'bagussssssssss': 'bagus', 'klh': 'kalah', 'setres': 'stres', 'ngaco': 'kacau',
    'hepi': 'senang', 'geme': 'game', 'ngelek': 'lag', 'boring': 'bosan'
}
normalize_word_dict.update(slangwords)

In [28]:
def cleaning(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) #remove URLs
    text = re.sub(r'[@#][A-Za-z0-9_]+', '', text) # remove mentions and hashtags
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text) # remove emojis
    text = re.sub(r'[\U00002000-\U00003000]', '', text) # remove special characters
    text = re.sub(r'[^\w\s]', '', text) # remove punctuation
    text = re.sub(r'\d+', '', text) # remove numbers
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces
    return text

def case_folding(text):
    return text.lower()

def normalization(text):
    for alay, baku in normalize_word_dict.items():
        text = re.sub(r'\b' + re.escape(alay) + r'\b', baku, text)
    return text

def tokenizing(text):
    return word_tokenize(text)

def stopwords_removal(words): # Remove common words that do not add significant meaning
    stop_words = set(stopwords.words('indonesian'))
    stop_words.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = [word for word in words if word not in stop_words]
    return filtered

factory = StemmerFactory()
stemmer = factory.create_stemmer()
def stemming(words): # Return word to the roots of words
    words = [stemmer.stem(word) for word in words]
    text = ' '.join(words)
    return text

def preprocess_text(text):
    text = cleaning(text)
    text = case_folding(text)
    text = normalization(text)
    words = tokenizing(text)
    words = stopwords_removal(words)
    text = stemming(words)
    return text

In [ ]:
df['clean_review'] = df['content'].apply(preprocess_text)
texts = df['clean_review'].tolist()

print(df[['content', 'clean_review']].head(10))

In [ ]:
df.drop(columns=['score'],inplace=True)
df

In [ ]:
df.info()

In [ ]:
df = df.dropna().reset_index(drop=True)
df

In [ ]:
df.info()

In [14]:
df.to_csv('clean_reviews.csv', index=False, encoding='utf-8')

# Dictionary

## Studi Literatur

In [15]:
# Kamus awal hasil studi literatur --> belum lengkap
effectiveness = [
    "jelas", "lengkap", "tujuan", "kesalahan", "bug", "macet", "glitch", "beku",
    "tersangkut", "menang", "memuat", "menerima", "mencapai", "menilai", "belajar",
    "latih", "masalah teknis", "mekanisme game", "lag", "masalah", "hancur"
]

satisfaction = [
    "puas", "kegunaan", "kepercayaan", "nyaman", "mudah", "belajar", "panduan",
    "motivasi", "pembaruan", "hadiah", "curang", "menyediakan", "terlibat",
    "menikmati", "seru", "setara", "seimbang", "membosankan"
]

## Ekstraksi Keyword

### TF-IDF

In [16]:
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['clean_review'])
tfidf_scores = X.mean(axis=0).A1
terms = vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame({'term': terms, 'score': tfidf_scores})
tfidf_df = tfidf_df.sort_values(by='score', ascending=False)
tfidf_df.head(20)

,term,score
830,game,0.073537
188,bagus,0.049821
1618,main,0.036457
2466,seru,0.035031
1085,hero,0.031023
2604,suka,0.018685
1317,kasih,0.017815
2781,tolong,0.017689
1656,mantap,0.016619
2507,sinergi,0.015486


### YAKE

In [17]:
import yake

df = pd.read_csv('clean_reviews.csv')
text = " ".join(df['clean_review'].astype(str))

# Simple usage with default parameters
kw_extractor = yake.KeywordExtractor()
keywords = kw_extractor.extract_keywords(text)

# With custom parameters
custom_kw_extractor = yake.KeywordExtractor(
    n=1,                   # ngram size
    dedupLim=0.9,          # deduplication threshold
    dedupFunc='seqm',      # deduplication function
    windowsSize=1,         # context window
    top=20,                # number of keywords to extract
)

keywords = custom_kw_extractor.extract_keywords(text)

sorted_keywords = sorted(keywords, key=lambda x: x[1], reverse=False)

for kw, score in keywords:
    print(f"{kw} ({score})")

game (1.2977415728911046e-06)
main (3.3415586561465006e-06)
hero (3.406540910542569e-06)
bagus (4.701257866965653e-06)
tolong (1.1410293737542867e-05)
seru (1.1609005068726334e-05)
kasih (1.523584594081841e-05)
bintang (1.592535098192131e-05)
sinergi (1.7658872821801785e-05)
kalah (2.6438472420940583e-05)
baik (3.079650791146396e-05)
magic (3.2906278914331436e-05)
update (3.298864179517995e-05)
susah (3.366511631039518e-05)
suka (3.5173584011890345e-05)
jaring (4.419717324392777e-05)
menang (4.661874735054584e-05)
gold (5.186050046891968e-05)
sistem (5.1917875074153e-05)
moonton (5.267787898432301e-05)


### KeyBERT

In [18]:
from keybert import KeyBERT

kw_model = KeyBERT(model='paraphrase-multilingual-mpnet-base-v2')

def extract_keywords_v2(text):
    if isinstance(text, str) and text.strip():
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),  # unigram + bigram
            top_n=10,
        )
        return [(kw, score) for kw, score in keywords if score > 0.3]
    return []

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
df['keywords_v2'] = df['clean_review'].apply(extract_keywords_v2)

In [ ]:
# Flatten semua keyword dan skor ke dalam satu list
all_keywords = []
for kw_list in df['keywords_v2']:
    for kw, score in kw_list:
        all_keywords.append((kw, score))

# Buat DataFrame dari hasil
kw_df = pd.DataFrame(all_keywords, columns=['keyword_v2', 'score'])

# Hitung rata-rata skor tiap keyword di seluruh dokumen
kw_summary = kw_df.groupby('keyword_v2', as_index=False)['score'].mean()

# Urutkan berdasarkan skor tertinggi
kw_summary = kw_summary.sort_values(by='score', ascending=False)

# Ambil 50 keyword dengan skor tertinggi
top_keywords = kw_summary.head(50)

# Tampilkan hasil
print(kw_summary.head(50))

## Ekspansi Kamus

In [ ]:
# from gensim.models import Word2Vec
# from gensim.utils import simple_preprocess

# # Tokenisasi sederhana
# sentences = [simple_preprocess(doc) for doc in docs]

# # Training Word2Vec model
# model_w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=1, sg=1)

# # Cek kata yang paling mirip dengan "menyenangkan" --> diganti dengan kamus aspek nanti
# similar_words = model_w2v.wv.most_similar("menyenangkan", topn=5)
# print(similar_words)

# #[('seru', 0.89), ('bagus', 0.85), ('baik', 0.82), ('menarik', 0.79), ('hebat', 0.78)]


# Pelabelan Sentimen

# Validasi Expert

# Pelabelan aspek

### TF IDF

In [ ]:
# satisfaction_keywords = ['senang', 'puas', 'bagus', 'baik', 'mantap', 'terbaik', 'lancar', 'cepat', 'nyaman',
#                          'mudah', 'rekomendasi', 'terpercaya', 'memuaskan', 'asik', 'seru', 'hebat', 'keren', 'top', 'oke', 'ok']

# effectiveness_keywords = ['efektif', 'efisien', 'berguna', 'bermanfaat', 'optimal', 'produktif', 'terampil', 'kompeten',
#                           'terorganisir', 'terstruktur', 'fokus', 'disiplin']

# def label_aspect(review):
#     review = str(review).lower().split()
#     satisfaction_count = sum(1 for word in review if word in satisfaction_keywords)
#     effectiveness_count = sum(1 for word in review if word in effectiveness_keywords)

#     if satisfaction_count > effectiveness_count:
#         aspect = 'Satisfaction'
#     elif effectiveness_count > satisfaction_count :
#         aspect = 'Effectiveness'
#     else:
#         aspect = 'None'

#     return pd.Series([aspect, effectiveness_count, satisfaction_count])

### KeyBERT

In [ ]:
# # fuzzy matching
# def match_keywords_to_categories(keywords, categories):
#     categories_score = {category: 0 for category in categories}
#     matches_details = {category: [] for category in categories}
#     for keyword, _ in keywords:
#         for category, target_keywords in categories.items():
#             best_match, score, _ = process.extractOne(
#                 keyword, target_keywords, scorer=fuzz.partial_ratio
#             )
#             if score > 80:
#                 categories_score[category] += score / 100.0
#                 matches_details[category].append((keyword, best_match, score))
#         return categories_score, matches_details

# def determine_aspect(categories_score):
#     if all(score == 0 for score in categories_score.values()):
#         return 'Unknown'
#     max_score = max(categories_score.value())
#     best_categories = [category for category, score in categories_score.items() if score == max_score]
#     return random.choice(best_categories)if len(best_categories) > 1 else best_categories[0]